In [41]:
import pandas as pd

veraset_file = '/mnt/disk/data/veraset_data_march.csv'
df = pd.read_csv(veraset_file)

print(df.shape)

(1074206, 21)


In [42]:
df.columns

Index(['Unnamed: 0', 'user_id', 'staypoint_id', 'latitude', 'longitude',
       'arrive_ts', 'leave_ts', 'spd_point', 'safegraph_place_id',
       'safegraph_place_id.1', 'location_name', 'naics_code', 'poi_latitude',
       'poi_longitude', 'street_address', 'area_square_feet', 'poi_polygon',
       'DistanceToPlaceCentroid_inkm', 'DistanceToPlaceWkt',
       'DistanceToPlaceCentroidRank', 'DistanceToPlaceWktRank'],
      dtype='object')

In [43]:
df['safegraph_place_id.1']

0          sg:64bdd157c5fc4c15a01c4b770ce4be41
1          sg:3f4392807bf84c769adebab067e3e16d
2          sg:c29dde02a5bf429f90a20143a5ce1ac5
3          sg:d08baf12f4f84341a3eca5a74c89c17e
4          sg:f1343cee56a9401b86824f8cc9e19f5f
                          ...                 
1074201    sg:1631ef3723db490d89b5777a649c8a43
1074202    sg:1631ef3723db490d89b5777a649c8a43
1074203    sg:835b260359a446828156ebbe9c4877f6
1074204    sg:835b260359a446828156ebbe9c4877f6
1074205    sg:360d88ef2ced4be180ea24290b9f9df4
Name: safegraph_place_id.1, Length: 1074206, dtype: object

In [51]:
import pandas as pd

# Paths to your files
file1 = '/mnt/disk/data/Houston_poi_all.csv'
file2 = '/mnt/disk/data/POI_data/Safegraph/safegraph_pois_Houston.csv'

# Load them
df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

In [52]:
merged = pd.merge(
    df1, df2,
    on='safegraph_place_id',
    how='outer',
    suffixes=('_df1', '_df2')
)

In [53]:
merged.columns

Index(['Unnamed: 0', 'safegraph_place_id', 'parent_safegraph_place_id_df1',
       'location_name_df1', 'safegraph_brand_ids_df1', 'brands_df1',
       'top_category_df1', 'sub_category_df1', 'naics_code_df1',
       'latitude_df1', 'longitude_df1', 'street_address_df1', 'city_df1',
       'region_df1', 'postal_code_df1', 'iso_country_code_df1',
       'phone_number_df1', 'open_hours_df1', 'category_tags_df1', 'placekey',
       'parent_placekey', 'parent_safegraph_place_id_df2',
       'safegraph_brand_ids_df2', 'location_name_df2', 'brands_df2',
       'top_category_df2', 'sub_category_df2', 'naics_code_df2',
       'latitude_df2', 'longitude_df2', 'street_address_df2', 'city_df2',
       'region_df2', 'postal_code_df2', 'open_hours_df2', 'category_tags_df2',
       'opened_on', 'closed_on', 'tracking_opened_since',
       'tracking_closed_since', 'polygon_wkt', 'polygon_class',
       'building_height', 'enclosed', 'phone_number_df2', 'is_synthetic',
       'includes_parking_lot', '

In [54]:
# Identify all attribute columns (except the key)
cols = [c for c in merged.columns if c not in ['safegraph_place_id']]

# Create a list of "base" columns (without _df1/_df2), i.e., unique attribute names
base_cols = set()
for c in cols:
    if c.endswith('_df1'):
        base_cols.add(c.replace('_df1',''))
    elif c.endswith('_df2'):
        base_cols.add(c.replace('_df2',''))
    else:
        base_cols.add(c)

# Now, for each row, fill in a new DataFrame with the "best" value per column
records = []
for _, row in merged.iterrows():
    rec = {'safegraph_place_id': row['safegraph_place_id']}
    for bc in base_cols:
        v1 = row.get(f"{bc}_df1", pd.NA)
        v2 = row.get(f"{bc}_df2", pd.NA)
        # Prefer df2 if present, else df1
        if pd.notna(v2):
            rec[bc] = v2
        elif pd.notna(v1):
            rec[bc] = v1
        else:
            # If only appears as an unsuffixed column
            rec[bc] = row.get(bc, pd.NA)
    records.append(rec)

# Build the cleaned DataFrame
poi_cleaned = pd.DataFrame.from_records(records)

In [55]:
poi_cleaned.shape

(73647, 31)

In [56]:
poi_cleaned.head()

,safegraph_place_id,postal_code,building_height,top_category,phone_number,enclosed,is_synthetic,polygon_class,parent_placekey,opened_on,...,brands,closed_on,street_address,location_name,tracking_opened_since,parent_safegraph_place_id,tracking_closed_since,iso_country_code,sub_category,Unnamed: 0
0,sg:0000f687dbd64a3b935ba0b8f49fed80,77084.0,5.73,Religious Organizations,<NA>,False,False,OWNED_POLYGON,NaN,NaN,...,<NA>,NaN,4710 Wild Bluebonnet Way,Greater Love Christian Church,NaN,<NA>,2019-07,US,Religious Organizations,NaN
1,sg:00011ea2089a4795874f987b8ae534fe,77017.0,4.21,Restaurants and Other Eating Places,17136431201.0,False,False,OWNED_POLYGON,223-222@8fc-8ct-cnq,NaN,...,<NA>,NaN,8456 Gulf Fwy,Oriental Gourmet,NaN,sg:56a08b0287a1481d9f53d0602c9ecd59,2019-07,US,Full-Service Restaurants,35905.0
2,sg:0001c91a2b9e408b8d8e39a01f36c7c3,77050.0,4.88,Grocery Stores,12817417794.0,False,False,NaN,NaN,NaN,...,<NA>,2020-01,12108 Homestead Rd,Am Mini Mart,NaN,<NA>,2019-07,US,Supermarkets and Other Grocery (except Conveni...,NaN
3,sg:0001f32a43cd47dc8ca86c34e553ceb5,77040.0,5.85,Other Miscellaneous Store Retailers,17139391222.0,False,False,OWNED_POLYGON,NaN,NaN,...,<NA>,NaN,7225 Langtry St,Business Productive,NaN,<NA>,2019-07,US,Tobacco Stores,33444.0
4,sg:00037b7318e54d06a921cc8f3eff5e71,77057.0,NaN,Personal Care Services,17134981222.0,False,True,OWNED_POLYGON,zzw-22c@8fc-fhr-6ff,NaN,...,<NA>,NaN,1305 S Voss Rd,Susanna Schultz,NaN,sg:cb0088c187c941679e6e4f0fa08e9f5a,2019-07,US,Other Personal Care Services,30214.0


In [57]:
# rename all columns in poi_cleaned with prefix 'safegraph.'
# drop Unnamed: 0 column if exists
if 'Unnamed: 0' in poi_cleaned.columns:
    poi_cleaned = poi_cleaned.drop(columns=['Unnamed: 0'])
poi_cleaned.columns = ['safegraph.' + col for col in poi_cleaned.columns]
poi_cleaned.head()

,safegraph.safegraph_place_id,safegraph.postal_code,safegraph.building_height,safegraph.top_category,safegraph.phone_number,safegraph.enclosed,safegraph.is_synthetic,safegraph.polygon_class,safegraph.parent_placekey,safegraph.opened_on,...,safegraph.placekey,safegraph.brands,safegraph.closed_on,safegraph.street_address,safegraph.location_name,safegraph.tracking_opened_since,safegraph.parent_safegraph_place_id,safegraph.tracking_closed_since,safegraph.iso_country_code,safegraph.sub_category
0,sg:0000f687dbd64a3b935ba0b8f49fed80,77084.0,5.73,Religious Organizations,<NA>,False,False,OWNED_POLYGON,NaN,NaN,...,229-222@8fc-rn4-ct9,<NA>,NaN,4710 Wild Bluebonnet Way,Greater Love Christian Church,NaN,<NA>,2019-07,US,Religious Organizations
1,sg:00011ea2089a4795874f987b8ae534fe,77017.0,4.21,Restaurants and Other Eating Places,17136431201.0,False,False,OWNED_POLYGON,223-222@8fc-8ct-cnq,NaN,...,226-223@8fc-8ct-cnq,<NA>,NaN,8456 Gulf Fwy,Oriental Gourmet,NaN,sg:56a08b0287a1481d9f53d0602c9ecd59,2019-07,US,Full-Service Restaurants
2,sg:0001c91a2b9e408b8d8e39a01f36c7c3,77050.0,4.88,Grocery Stores,12817417794.0,False,False,NaN,NaN,NaN,...,224-222@8fc-9yn-pvz,<NA>,2020-01,12108 Homestead Rd,Am Mini Mart,NaN,<NA>,2019-07,US,Supermarkets and Other Grocery (except Conveni...
3,sg:0001f32a43cd47dc8ca86c34e553ceb5,77040.0,5.85,Other Miscellaneous Store Retailers,17139391222.0,False,False,OWNED_POLYGON,NaN,NaN,...,zzw-222@8fc-fgs-q4v,<NA>,NaN,7225 Langtry St,Business Productive,NaN,<NA>,2019-07,US,Tobacco Stores
4,sg:00037b7318e54d06a921cc8f3eff5e71,77057.0,NaN,Personal Care Services,17134981222.0,False,True,OWNED_POLYGON,zzw-22c@8fc-fhr-6ff,NaN,...,zzw-227@8fc-fhr-6ff,<NA>,NaN,1305 S Voss Rd,Susanna Schultz,NaN,sg:cb0088c187c941679e6e4f0fa08e9f5a,2019-07,US,Other Personal Care Services


In [58]:
poi_cleaned = poi_cleaned.rename(columns={
    "safegraph.safegraph_place_id": "safegraph.place_id",
    "safegraph.safegraph_brand_ids": "safegraph.brand_ids",
    "safegraph.parent_safegraph_place_id": "safegraph.parent_place_id"
})

In [59]:
# match safegraph_place_id in veraset with safegraph poi data # use safegraph. as prefix
merged_df = df.merge(poi_cleaned, left_on='safegraph_place_id.1', right_on='safegraph.place_id', how='left')
print(merged_df.shape)

(1074206, 51)


In [60]:
merged_df.columns

Index(['Unnamed: 0', 'user_id', 'staypoint_id', 'latitude', 'longitude',
       'arrive_ts', 'leave_ts', 'spd_point', 'safegraph_place_id',
       'safegraph_place_id.1', 'location_name', 'naics_code', 'poi_latitude',
       'poi_longitude', 'street_address', 'area_square_feet', 'poi_polygon',
       'DistanceToPlaceCentroid_inkm', 'DistanceToPlaceWkt',
       'DistanceToPlaceCentroidRank', 'DistanceToPlaceWktRank',
       'safegraph.place_id', 'safegraph.postal_code',
       'safegraph.building_height', 'safegraph.top_category',
       'safegraph.phone_number', 'safegraph.enclosed',
       'safegraph.is_synthetic', 'safegraph.polygon_class',
       'safegraph.parent_placekey', 'safegraph.opened_on',
       'safegraph.open_hours', 'safegraph.region', 'safegraph.brand_ids',
       'safegraph.polygon_wkt', 'safegraph.city',
       'safegraph.includes_parking_lot', 'safegraph.category_tags',
       'safegraph.naics_code', 'safegraph.latitude', 'safegraph.longitude',
       'safegraph.plac

In [61]:
merged_df = merged_df.rename(columns={
    "arrive_ts": "properties.started_at",
    "leave_ts": "properties.finished_at",
    "user_id": "properties.user_id"
})

## Preprocess data

In [62]:
merged_df[['properties.started_at', 'properties.finished_at']] = merged_df[['properties.started_at', 'properties.finished_at']].apply(pd.to_datetime, unit='s')

print(f"Veraset data time range: {merged_df['properties.started_at'].min()} - {merged_df['properties.finished_at'].max()}")

Veraset data time range: 2020-03-05 00:00:01 - 2020-03-26 23:59:59


In [63]:
print("Veraset has {} unique POIs.".format(merged_df['safegraph.place_id'].nunique()))

Veraset has 32159 unique POIs.


In [64]:
print("Veraset has {} unique users.".format(merged_df['properties.user_id'].nunique()))

Veraset has 224199 unique users.


In [65]:
merged_df['properties.user_id'].min(), merged_df['properties.user_id'].max()

(np.int64(6), np.int64(2240935))

In [66]:
# group all visits by user_id
user_visits = merged_df.groupby('properties.user_id').size().reset_index(name='visits')

# get stats on total visits per user
user_visits['visits'].describe()

count    224199.000000
mean          4.791306
std           7.180159
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         238.000000
Name: visits, dtype: float64

In [67]:
# drop users with less than 5 visits
user_visits = user_visits[user_visits['visits'] > 5]

# keep the visits of the selected users in df
merged_df = merged_df[merged_df['properties.user_id'].isin(user_visits['properties.user_id'])]

In [68]:
# cross check the stats again
user_visits = merged_df.groupby('properties.user_id').size().reset_index(name='visits')
user_visits['visits'].describe()

count    52999.000000
mean        13.502217
std         10.657113
min          6.000000
25%          7.000000
50%         10.000000
75%         16.000000
max        238.000000
Name: visits, dtype: float64

In [69]:
print("Veraset has {} unique POIs after removing small sequencies.".format(merged_df['safegraph.place_id'].nunique()))

Veraset has 28419 unique POIs after removing small sequencies.


In [70]:
# map user_id to a continuous range of integers starting from 0
user_id_mapping = {old_id: new_id for new_id, old_id in enumerate(merged_df['properties.user_id'].unique())}
merged_df['user_id'] = merged_df['properties.user_id'].map(user_id_mapping)

In [71]:
merged_df['user_id'].min(), merged_df['user_id'].max()

(np.int64(0), np.int64(52998))

In [72]:
merged_df['place_id'] = merged_df['safegraph.place_id'].astype('category').cat.codes

merged_df['place_id'].min(), merged_df['place_id'].max()

(np.int16(0), np.int16(28418))

In [73]:
merged_df['category'] = merged_df['safegraph.top_category'].astype('category').cat.codes + 1  # some POIs do not have category

print(merged_df['category'].min(), merged_df['category'].max())

0 149


In [74]:
# count number of POIs with category 0
print("Number of POIs with category 0: {}".format((merged_df['category'] == 0).sum()))

Number of POIs with category 0: 4578


In [75]:
# pick 1 POI with category 0 and print the other attributes
print(merged_df[merged_df['category'] == 0].iloc[0])

Unnamed: 0                                                                       554
properties.user_id                                                               390
staypoint_id                                                                    2810
latitude                                                                   29.686968
longitude                                                                 -95.405518
properties.started_at                                            2020-03-10 03:39:04
properties.finished_at                                           2020-03-10 04:23:45
spd_point                                             POINT (-95.4055185 29.6869685)
safegraph_place_id                                                           31474.0
safegraph_place_id.1                             sg:c3a227d498df4fae90fef8790e62e573
location_name                                                             Nrg Center
naics_code                                                       

In [76]:
merged_df['category'].nunique()

150

In [77]:
merged_df['naics_2digit'] = merged_df['safegraph.naics_code'].apply(
    lambda x: str(int(x))[:2] if pd.notnull(x) else None
)

In [78]:
merged_df['naics_2digit'].unique()

array(['81', '71', '44', '72', '49', '51', '62', '45', '53', '52', '61',
       '54', '48', None, '56', '23', '31', '33', '92', '42', '32', '22'],
      dtype=object)

In [79]:
merged_df['top_naics_category'] = merged_df['naics_2digit'].astype('category').cat.codes

# replace -1 with max + 1
merged_df['top_naics_category'] = merged_df['top_naics_category'] + 1

In [80]:
merged_df['top_naics_category'].value_counts()

top_naics_category
13    129404
7     101496
17     95371
18     87674
19     76727
16     58445
20     39008
12     33436
8      26016
9      20326
14     10361
21      7935
10      6842
11      4866
15      4790
0       4578
3       2710
6       2577
2       1572
5        735
4        724
1         11
Name: count, dtype: int64

In [81]:
# create place_lat and place_lon columns. Assign safegraph.latitude and safegraph.longitude to them if not nan else assign geometry.latitude and geometry.longitude

merged_df['place_lat'] = merged_df['safegraph.latitude'].where(
    merged_df['safegraph.latitude'].notnull(), merged_df['poi_latitude']
)

merged_df['place_lon'] = merged_df['safegraph.longitude'].where(
    merged_df['safegraph.longitude'].notnull(), merged_df['poi_longitude']
)

In [82]:
merged_df['place_lat'].isnull().sum(), merged_df['place_lon'].isnull().sum()

(np.int64(0), np.int64(0))

In [83]:
merged_df = merged_df.rename(columns={
    'properties.started_at': 'arrival_time',
    'properties.finished_at': 'departure_time',
})

In [84]:
merged_df.columns

Index(['Unnamed: 0', 'properties.user_id', 'staypoint_id', 'latitude',
       'longitude', 'arrival_time', 'departure_time', 'spd_point',
       'safegraph_place_id', 'safegraph_place_id.1', 'location_name',
       'naics_code', 'poi_latitude', 'poi_longitude', 'street_address',
       'area_square_feet', 'poi_polygon', 'DistanceToPlaceCentroid_inkm',
       'DistanceToPlaceWkt', 'DistanceToPlaceCentroidRank',
       'DistanceToPlaceWktRank', 'safegraph.place_id', 'safegraph.postal_code',
       'safegraph.building_height', 'safegraph.top_category',
       'safegraph.phone_number', 'safegraph.enclosed',
       'safegraph.is_synthetic', 'safegraph.polygon_class',
       'safegraph.parent_placekey', 'safegraph.opened_on',
       'safegraph.open_hours', 'safegraph.region', 'safegraph.brand_ids',
       'safegraph.polygon_wkt', 'safegraph.city',
       'safegraph.includes_parking_lot', 'safegraph.category_tags',
       'safegraph.naics_code', 'safegraph.latitude', 'safegraph.longitude',
  

In [85]:
merged_df.to_parquet("/mnt/disk/data/trajfm_veraset_splits/veraset/Visits/Houston/whole_veraset_processed.parquet", index=False)

In [86]:
# compute bbox of the area
min_lat, min_lon = merged_df['place_lat'].min(), merged_df['place_lon'].min()
max_lat, max_lon = merged_df['place_lat'].max(), merged_df['place_lon'].max()
min_lat, min_lon, max_lat, max_lon

(np.float64(29.549943),
 np.float64(-95.558428),
 np.float64(29.951268),
 np.float64(-95.158462))

## Save poi_cleaned data

In [92]:
# rename safegraph.place_id to safegraph_place_id
poi_cleaned = poi_cleaned.rename(columns={"safegraph.place_id": "safegraph_place_id"})

# add latitude and longitude columns to poi_cleaned
poi_cleaned['latitude'] = poi_cleaned['safegraph.latitude']
poi_cleaned['longitude'] = poi_cleaned['safegraph.longitude']

In [93]:
poi_cleaned.columns

Index(['safegraph_place_id', 'safegraph.postal_code',
       'safegraph.building_height', 'safegraph.top_category',
       'safegraph.phone_number', 'safegraph.enclosed',
       'safegraph.is_synthetic', 'safegraph.polygon_class',
       'safegraph.parent_placekey', 'safegraph.opened_on',
       'safegraph.open_hours', 'safegraph.region', 'safegraph.brand_ids',
       'safegraph.polygon_wkt', 'safegraph.city',
       'safegraph.includes_parking_lot', 'safegraph.category_tags',
       'safegraph.naics_code', 'safegraph.latitude', 'safegraph.longitude',
       'safegraph.placekey', 'safegraph.brands', 'safegraph.closed_on',
       'safegraph.street_address', 'safegraph.location_name',
       'safegraph.tracking_opened_since', 'safegraph.parent_place_id',
       'safegraph.tracking_closed_since', 'safegraph.iso_country_code',
       'safegraph.sub_category', 'latitude', 'longitude'],
      dtype='object')

In [95]:
# save the cleaned poi data
poi_cleaned.to_csv("/mnt/disk/data/POI_data/Safegraph/safegraph_pois_Houston_cleaned.csv", index=False)

In [96]:
poi_cleaned.columns

Index(['safegraph_place_id', 'safegraph.postal_code',
       'safegraph.building_height', 'safegraph.top_category',
       'safegraph.phone_number', 'safegraph.enclosed',
       'safegraph.is_synthetic', 'safegraph.polygon_class',
       'safegraph.parent_placekey', 'safegraph.opened_on',
       'safegraph.open_hours', 'safegraph.region', 'safegraph.brand_ids',
       'safegraph.polygon_wkt', 'safegraph.city',
       'safegraph.includes_parking_lot', 'safegraph.category_tags',
       'safegraph.naics_code', 'safegraph.latitude', 'safegraph.longitude',
       'safegraph.placekey', 'safegraph.brands', 'safegraph.closed_on',
       'safegraph.street_address', 'safegraph.location_name',
       'safegraph.tracking_opened_since', 'safegraph.parent_place_id',
       'safegraph.tracking_closed_since', 'safegraph.iso_country_code',
       'safegraph.sub_category', 'latitude', 'longitude'],
      dtype='object')

In [97]:
poi_cleaned['safegraph_place_id'].nunique(), merged_df['safegraph.place_id'].nunique()

(73647, 28419)